# Request API Weather


In [28]:
import requests
import os
import pandas as pd
import time

In [29]:
country_code = "FR"
# Liste des villes
villes = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen",
    "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon", "Annecy", "Grenoble",
    "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis",
    "Marseille", "Aix en Provence", "Avignon", "Uzes", "Nimes",
    "Aigues Mortes", "Saintes Maries de la mer", "Collioure",
    "Carcassonne", "Ariege", "Toulouse", "Montauban", "Biarritz",
    "Bayonne", "La Rochelle"
]

In [ ]:
API_key = os.getenv("Weather_API_key")
resultats = []

for ville in villes:
    # 1ère requête : Géocodage (nom → coordonnées)
    url_geo = f"http://api.openweathermap.org/geo/1.0/direct?q={ville},FR&limit=1&appid={API_key}"
    r_geo = requests.get(url_geo)
    
    if r_geo.status_code == 200 and r_geo.json():
        geo_data = r_geo.json()[0]
        lat = geo_data['lat']
        lon = geo_data['lon']
        cnt = 7  # Nombre de jours pour la météo
        print(f"✓ {ville} : lat={lat}, lon={lon}")
        
        # 2ème requête : Météo (coordonnées → météo)
        url_meteo = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_key}&units=metric"
        #api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_key}  météo actuelle gratuite
        #api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API key}  prévisions 5 jours gratuites
        #api.openweathermap.org/data/2.5/forecast/daily?lat={lat}&lon={lon}&cnt={cnt}&appid={API_key}   16 jours nb =cnt abonnement payant
        r_meteo = requests.get(url_meteo)
        
        if r_meteo.status_code == 200:
            resultats.append(r_meteo.json())
            print(f"  → {r_meteo.json()['main']['temp']}°C - {r_meteo.json()['weather'][0]['description']}")
        else:
            print(f"  ✗ Erreur météo : {r_meteo.status_code}")
    else:
        print(f"✗ Erreur de géocodage pour {ville} : {r_geo.status_code}")
    
    time.sleep(1)  # Respect limite API



✓ Mont Saint Michel : lat=48.6359541, lon=-1.511459954959514
  → 2.69°C - scattered clouds
✓ St Malo : lat=48.649518, lon=-2.0260409
  → 5.29°C - clear sky
✓ Bayeux : lat=49.2764624, lon=-0.7024738
  → 6.14°C - broken clouds
✓ Le Havre : lat=49.4938975, lon=0.1079732
  → 2.51°C - clear sky
✓ Rouen : lat=49.4404591, lon=1.0939658
  → 4.3°C - clear sky
✓ Paris : lat=48.8588897, lon=2.3200410217200766
  → 4.27°C - clear sky
✓ Amiens : lat=49.8941708, lon=2.2956951
  → 2.83°C - few clouds
✓ Lille : lat=50.6365654, lon=3.0635282
  → 4.19°C - clear sky
✓ Strasbourg : lat=48.584614, lon=7.7507127
  → 0.95°C - clear sky
✓ Chateau du Haut Koenigsbourg : lat=48.2495226, lon=7.3454923
  → -2.1°C - clear sky
✓ Colmar : lat=48.0777517, lon=7.3579641
  → 0.91°C - clear sky
✓ Eguisheim : lat=48.0447968, lon=7.3079618
  → 0.84°C - clear sky
✓ Besancon : lat=47.2380222, lon=6.0243622
  → 0.8°C - clear sky
✓ Dijon : lat=47.3215806, lon=5.0414701
  → 1.8°C - few clouds
✓ Annecy : lat=45.8992348, lon=6.12

In [43]:
# DataFrame final
df = pd.json_normalize(resultats)
weather = pd.json_normalize(resultats, record_path='weather')
df = df.join(weather, lsuffix=' ', rsuffix='_weather')
df.drop(columns=['weather'], inplace=True)



In [44]:
df.to_csv("Meteo_villes_france2.csv", index=True)
print(f"\n✓ Total : {len(df)} villes sauvegardées")


✓ Total : 34 villes sauvegardées
